# E0 — RGB-only Baseline

This notebook contains only the E0 RGB-only experiment.

It uses the validated RGB/VIS pairs from the frozen multimodal split.

**Split**
- Development: 4 collection-context groups
- Final test: `210327_Airfield_FLIR` and `210812_Hannegan_Enterprise`
- Development evaluation: 4-fold grouped cross-validation
- No image-level random split
- Final test contexts remain untouched until the final evaluation

**Model**
- YOLOv8s pretrained detector
- One class: `person`

**Validated pairs**
- Total: 10,703
- Development: 9,096
- Final test: 1,607


In [ ]:
import sys
import subprocess

# Install the detector package in the current Colab kernel.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"])


In [ ]:
from pathlib import Path
import os
import re
import json
import shutil
import zipfile
import hashlib
from collections import Counter

import pandas as pd
import numpy as np

from IPython.display import display

import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available. Training will be very slow.")


In [ ]:
# Mount Google Drive in Colab.
from google.colab import drive
drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/WiSARD/WiSARDv1.zip")

PROJECT_ROOT = Path("/content/UAV_SAR")
RESULTS_ROOT = PROJECT_ROOT / "results" / "rgb_baseline"
DATA_ROOT = PROJECT_ROOT / "data" / "rgb_baseline"
MANIFEST_ROOT = PROJECT_ROOT / "results" / "dataset_split"

SPLIT_MANIFEST = MANIFEST_ROOT / "recording_split_manifest.csv"
PAIR_MANIFEST_CANDIDATES = [
    PROJECT_ROOT / "results" / "rgb_baseline" / "rgb_pair_manifest.csv",
    PROJECT_ROOT / "results" / "rgb" / "rgb_pair_manifest.csv",
    Path("/results/rgb_baseline/rgb_pair_manifest.csv"),
]

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("ZIP:", ZIP_PATH)
print("Split manifest:", SPLIT_MANIFEST)


In [ ]:
# Locate the validated RGB pair manifest.
PAIR_MANIFEST = next((p for p in PAIR_MANIFEST_CANDIDATES if p.exists()), None)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"WiSARDv1.zip not found: {ZIP_PATH}")

if not SPLIT_MANIFEST.exists():
    raise FileNotFoundError(
        f"Frozen split manifest not found: {SPLIT_MANIFEST}. "
        "Copy the frozen manifest into this path before running E0."
    )

if PAIR_MANIFEST is None:
    raise FileNotFoundError(
        "Validated RGB pair manifest not found. Expected one of:\n" +
        "\n".join(str(p) for p in PAIR_MANIFEST_CANDIDATES)
    )

print("Using pair manifest:", PAIR_MANIFEST)


In [ ]:
# Load the two authoritative manifests.
split_df = pd.read_csv(SPLIT_MANIFEST)
pairs_df = pd.read_csv(PAIR_MANIFEST)

print("Frozen split columns:")
print(list(split_df.columns))
print("\nRGB pair manifest columns:")
print(list(pairs_df.columns))

display(split_df.head())
display(pairs_df.head())


In [ ]:
# Find the important columns without guessing their values.
def find_column(df, names, required=True):
    normalized = {str(c).strip().lower(): c for c in df.columns}
    for name in names:
        if name.lower() in normalized:
            return normalized[name.lower()]
    if required:
        raise KeyError(f"Could not find any of {names}. Available columns: {list(df.columns)}")
    return None

split_recording_col = find_column(
    split_df,
    ["recording", "recording_name", "recording_folder", "recording_id"]
)

split_partition_col = find_column(
    split_df,
    ["partition", "split"]
)

pair_image_col = find_column(
    pairs_df,
    ["image_member", "image_path", "image", "rgb_image_member"]
)

pair_label_col = find_column(
    pairs_df,
    ["annotation_member", "label_member", "label_path", "annotation", "label"]
)

pair_recording_col = find_column(
    pairs_df,
    ["recording", "recording_name", "recording_folder", "recording_id"]
)

pair_partition_col = find_column(
    pairs_df,
    ["partition", "split"]
)

# Keep only the fields needed for E0.
pairs = pairs_df[
    [pair_image_col, pair_label_col, pair_recording_col, pair_partition_col]
].copy()

pairs.columns = ["image_member", "label_member", "recording", "partition"]

print(pairs.dtypes)


In [ ]:
# Frozen split definitions.
TEST_CONTEXTS = {
    "210327_Airfield_FLIR",
    "210812_Hannegan_Enterprise",
}

DEVELOPMENT_CONTEXTS = {
    "210417_MtErie_Enterprise",
    "210529_Carnation_Enterprise",
    "210924_FHL_Enterprise",
    "220109_Baker_Enterprise",
}

EXPECTED_TOTAL = 10_703
EXPECTED_DEVELOPMENT = 9_096
EXPECTED_TEST = 1_607

# Map each recording to its frozen partition.
split_lookup = split_df[[split_recording_col, split_partition_col]].copy()
split_lookup.columns = ["recording", "frozen_partition"]

pairs = pairs.drop(columns=["partition"]).merge(
    split_lookup,
    on="recording",
    how="left",
    validate="many_to_one",
)

if pairs["frozen_partition"].isna().any():
    missing = pairs.loc[pairs["frozen_partition"].isna(), "recording"].unique().tolist()
    raise ValueError(f"Pair manifest contains recordings missing from frozen split: {missing}")

pairs["partition"] = pairs["frozen_partition"]
pairs = pairs.drop(columns=["frozen_partition"])

# The validated preparation stage must already have excluded unannotated and invalid-box pairs.
if len(pairs) != EXPECTED_TOTAL:
    raise AssertionError(
        f"Expected exactly {EXPECTED_TOTAL} validated RGB pairs, found {len(pairs)}"
    )

development_count = int((pairs["partition"] == "development").sum())
test_count = int((pairs["partition"] == "test").sum())

if development_count != EXPECTED_DEVELOPMENT:
    raise AssertionError(
        f"Expected {EXPECTED_DEVELOPMENT} development pairs, found {development_count}"
    )

if test_count != EXPECTED_TEST:
    raise AssertionError(
        f"Expected {EXPECTED_TEST} test pairs, found {test_count}"
    )

# Confirm the frozen context boundary.
def context_from_recording(recording):
    for context in TEST_CONTEXTS | DEVELOPMENT_CONTEXTS:
        if str(recording).startswith(context + "_"):
            return context
    return None

pairs["context"] = pairs["recording"].map(context_from_recording)

if pairs["context"].isna().any():
    bad = pairs.loc[pairs["context"].isna(), "recording"].unique().tolist()
    raise AssertionError(f"Could not map these recordings to frozen multimodal contexts: {bad}")

if set(pairs.loc[pairs["partition"] == "test", "context"]) != TEST_CONTEXTS:
    raise AssertionError("Final test contexts do not match the frozen test definition.")

if set(pairs.loc[pairs["partition"] == "development", "context"]) != DEVELOPMENT_CONTEXTS:
    raise AssertionError("Development contexts do not match the frozen development definition.")

print("Validated RGB pairs:", len(pairs))
print("Development:", development_count)
print("Final test:", test_count)
print("Development contexts:", sorted(DEVELOPMENT_CONTEXTS))
print("Test contexts:", sorted(TEST_CONTEXTS))
display(pairs.groupby(["partition", "context"]).size().rename("pairs").reset_index())


In [ ]:
# Build a compact model-ready workspace from the ZIP.
# Only the 10,703 validated RGB image/label pairs are extracted.
#
# The original WiSARD ZIP remains untouched.

WORKSPACE = DATA_ROOT / "workspace"
IMAGE_POOL = WORKSPACE / "images"
LABEL_POOL = WORKSPACE / "labels"

WORKSPACE.mkdir(parents=True, exist_ok=True)
IMAGE_POOL.mkdir(parents=True, exist_ok=True)
LABEL_POOL.mkdir(parents=True, exist_ok=True)

def safe_id(text):
    text = str(text)
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text)

# Use a deterministic filename based on the original recording + basename.
pairs["image_id"] = pairs.apply(
    lambda r: f"{safe_id(r.recording)}__{Path(r.image_member).stem}",
    axis=1
)

pairs["local_image"] = pairs["image_id"].map(lambda x: IMAGE_POOL / f"{x}.jpeg")
pairs["local_label"] = pairs["image_id"].map(lambda x: LABEL_POOL / f"{x}.txt")

missing_images = []
missing_labels = []

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    members = set(zf.namelist())

    for row in pairs.itertuples(index=False):
        if row.image_member not in members:
            missing_images.append(row.image_member)
        if row.label_member not in members:
            missing_labels.append(row.label_member)

    if missing_images or missing_labels:
        raise FileNotFoundError(
            f"ZIP member check failed. Missing images={len(missing_images)}, "
            f"missing labels={len(missing_labels)}"
        )

    for row in pairs.itertuples(index=False):
        image_path = Path(row.local_image)
        label_path = Path(row.local_label)

        if not image_path.exists():
            image_path.write_bytes(zf.read(row.image_member))

        if not label_path.exists():
            label_path.write_bytes(zf.read(row.label_member))

print("Workspace ready.")
print("Images:", len(list(IMAGE_POOL.glob("*.jpeg"))))
print("Labels:", len(list(LABEL_POOL.glob("*.txt"))))


In [ ]:
# Validate YOLO label format and class distribution.

bad_rows = []
class_counts = Counter()

for row in pairs.itertuples(index=False):
    label_path = Path(row.local_label)
    lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]

    for line_no, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            bad_rows.append((row.local_label, line_no, "wrong_field_count", line))
            continue

        try:
            cls = int(parts[0])
            vals = [float(x) for x in parts[1:]]
        except ValueError:
            bad_rows.append((row.local_label, line_no, "non_numeric", line))
            continue

        x, y, w, h = vals

        if cls != 0:
            bad_rows.append((row.local_label, line_no, "unexpected_class", line))
        if not all(0.0 <= v <= 1.0 for v in vals):
            bad_rows.append((row.local_label, line_no, "out_of_range", line))
        if w <= 0 or h <= 0:
            bad_rows.append((row.local_label, line_no, "non_positive_size", line))

        class_counts[cls] += 1

if bad_rows:
    print("First invalid labels:")
    for item in bad_rows[:10]:
        print(item)
    raise AssertionError(f"Found {len(bad_rows)} invalid YOLO label rows.")

print("YOLO label validation: PASS")
print("Class distribution:", dict(class_counts))


In [ ]:
# Define the 4 grouped development folds.
#
# Each development context becomes validation exactly once.
# No recording/context can appear in both train and validation within a fold.

DEV_CONTEXTS_ORDER = sorted(DEVELOPMENT_CONTEXTS)

folds = {}
for i, val_context in enumerate(DEV_CONTEXTS_ORDER, start=1):
    train_contexts = [c for c in DEV_CONTEXTS_ORDER if c != val_context]
    folds[i] = {
        "validation": val_context,
        "train": train_contexts,
    }

fold_table = pd.DataFrame([
    {
        "fold": fold_id,
        "train_contexts": ", ".join(spec["train"]),
        "validation_context": spec["validation"],
    }
    for fold_id, spec in folds.items()
])

display(fold_table)


In [ ]:
# Build the final training dataset from development contexts only.
#
# The final test contexts are kept completely outside the training loop.
# The final model is evaluated on them only after training is finished.

FINAL_ROOT = WORKSPACE / "final"
if FINAL_ROOT.exists():
    shutil.rmtree(FINAL_ROOT)

final_train_rows = pairs[pairs["partition"] == "development"].copy()
final_test_rows = pairs[pairs["partition"] == "test"].copy()

link_dataset(final_train_rows, FINAL_ROOT / "train")

# A test directory is created only for the post-training evaluation step.
link_dataset(final_test_rows, FINAL_ROOT / "test")

# YOLO needs a dataset configuration for training. The final training run
# explicitly disables validation, so the test set is never passed to train().
FINAL_DATA_YAML = FINAL_ROOT / "data.yaml"
write_yaml(
    FINAL_DATA_YAML,
    FINAL_ROOT / "train" / "images",
    FINAL_ROOT / "train" / "images",
    FINAL_ROOT / "test" / "images",
)

print("Final training pairs:", len(final_train_rows))
print("Final test pairs:", len(final_test_rows))


## Training configuration

The following cell controls the E0 run.

The workflow is:
1. Four grouped development folds for model/development evaluation.
2. Fixed training configuration after development evaluation.
3. Final fit on all development pairs with validation disabled.
4. One final evaluation on the two untouched test contexts.

The final test set is never used during training or hyperparameter selection.


In [ ]:
# E0 training configuration.

SEED = 42
MODEL_WEIGHTS = "yolov8s.pt"

EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 2

DEVICE = 0 if torch.cuda.is_available() else "cpu"

RUN_GROUPED_CV = True
RUN_FINAL_TRAINING = True

TRAIN_PROJECT = RESULTS_ROOT / "runs"
TRAIN_PROJECT.mkdir(parents=True, exist_ok=True)

print({
    "model": MODEL_WEIGHTS,
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "workers": WORKERS,
    "device": DEVICE,
    "seed": SEED,
})


In [ ]:
# Train the 4 grouped CV folds.

cv_results = []

if RUN_GROUPED_CV:
    for fold_id, spec in folds.items():
        print("=" * 80)
        print(f"TRAINING FOLD {fold_id}")
        print(f"Validation context: {spec['validation']}")
        print("=" * 80)

        model = YOLO(MODEL_WEIGHTS)

        fold_root = FOLD_ROOT / f"fold_{fold_id}"
        run_name = f"e0_rgb_fold_{fold_id}"

        model.train(
            data=str(fold_root / "data.yaml"),
            epochs=EPOCHS,
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            workers=WORKERS,
            device=DEVICE,
            seed=SEED,
            deterministic=True,
            project=str(TRAIN_PROJECT),
            name=run_name,
            exist_ok=True,
            pretrained=True,
            verbose=True,
        )

        metrics = model.val(
            data=str(fold_root / "data.yaml"),
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            workers=WORKERS,
            split="val",
        )

        cv_results.append({
            "fold": fold_id,
            "validation_context": spec["validation"],
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50_95": float(metrics.box.map),
        })

cv_results_df = pd.DataFrame(cv_results)

if not cv_results_df.empty:
    display(cv_results_df)
    print("\nCV mean:")
    display(
        cv_results_df[
            ["precision", "recall", "mAP50", "mAP50_95"]
        ].mean().to_frame("mean").T
    )


In [ ]:
# Save grouped-CV results.

if not cv_results_df.empty:
    cv_results_path = RESULTS_ROOT / "e0_rgb_grouped_cv_metrics.csv"
    cv_results_df.to_csv(cv_results_path, index=False)

    cv_summary = {
        "experiment": "E0_RGB_only",
        "model": MODEL_WEIGHTS,
        "epochs": EPOCHS,
        "imgsz": IMAGE_SIZE,
        "batch": BATCH_SIZE,
        "seed": SEED,
        "development_pairs": EXPECTED_DEVELOPMENT,
        "test_pairs": EXPECTED_TEST,
        "folds": len(folds),
        "cv_mean": {
            "precision": float(cv_results_df["precision"].mean()),
            "recall": float(cv_results_df["recall"].mean()),
            "mAP50": float(cv_results_df["mAP50"].mean()),
            "mAP50_95": float(cv_results_df["mAP50_95"].mean()),
        },
    }

    (RESULTS_ROOT / "e0_rgb_grouped_cv_summary.json").write_text(
        json.dumps(cv_summary, indent=2),
        encoding="utf-8",
    )

    print("Saved:", cv_results_path)


In [ ]:
# Final E0 model.
#
# Hyperparameters are fixed from the development-stage procedure above.
# Training uses ALL development pairs.
# Validation is disabled during this final fit.
# The frozen test contexts are evaluated only after training is complete.

final_model = None
final_metrics = None

if RUN_FINAL_TRAINING:
    final_model = YOLO(MODEL_WEIGHTS)

    final_model.train(
        data=str(FINAL_DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        workers=WORKERS,
        device=DEVICE,
        seed=SEED,
        deterministic=True,
        project=str(TRAIN_PROJECT),
        name="e0_rgb_final",
        exist_ok=True,
        pretrained=True,
        val=False,
        verbose=True,
    )

    # Use the final saved weights for the one-time test evaluation.
    final_weights = Path(TRAIN_PROJECT) / "e0_rgb_final" / "weights" / "last.pt"

    if not final_weights.exists():
        raise FileNotFoundError(
            f"Final training completed but expected weights were not found: {final_weights}"
        )

    final_model = YOLO(str(final_weights))

    test_data_yaml = FINAL_ROOT / "test_data.yaml"
    write_yaml(
        test_data_yaml,
        FINAL_ROOT / "test" / "images",
        FINAL_ROOT / "test" / "images",
    )

    final_metrics = final_model.val(
        data=str(test_data_yaml),
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        split="val",
    )

    final_test_metrics = pd.DataFrame([{
        "experiment": "E0_RGB_only",
        "precision": float(final_metrics.box.mp),
        "recall": float(final_metrics.box.mr),
        "mAP50": float(final_metrics.box.map50),
        "mAP50_95": float(final_metrics.box.map),
    }])

    display(final_test_metrics)

    final_test_metrics.to_csv(
        RESULTS_ROOT / "e0_rgb_final_test_metrics.csv",
        index=False,
    )

    print("Final test evaluation completed on untouched test contexts.")


In [ ]:
# Final prediction examples on the untouched test contexts.

if final_model is not None:
    sample_rows = final_test_rows.sample(
        n=min(12, len(final_test_rows)),
        random_state=SEED,
    )

    prediction_rows = []

    for row in sample_rows.itertuples(index=False):
        results = final_model.predict(
            source=str(row.local_image),
            imgsz=IMAGE_SIZE,
            conf=0.25,
            device=DEVICE,
            save=True,
            project=str(RESULTS_ROOT / "predictions"),
            name="test_examples",
            exist_ok=True,
            verbose=False,
        )

        prediction_rows.append({
            "image": Path(row.local_image).name,
            "context": row.context,
            "recording": row.recording,
            "prediction_count": len(results[0].boxes),
        })

    display(pd.DataFrame(prediction_rows))


In [ ]:
# Save the frozen E0 configuration and a compact result summary.

config = {
    "experiment": "E0_RGB_only",
    "model": MODEL_WEIGHTS,
    "seed": SEED,
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "workers": WORKERS,
    "device": str(DEVICE),
    "total_validated_rgb_pairs": EXPECTED_TOTAL,
    "development_pairs": EXPECTED_DEVELOPMENT,
    "test_pairs": EXPECTED_TEST,
    "development_contexts": sorted(DEVELOPMENT_CONTEXTS),
    "test_contexts": sorted(TEST_CONTEXTS),
    "grouped_cv_folds": folds,
}

(RESULTS_ROOT / "e0_rgb_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

summary_lines = [
    "# E0 RGB-only baseline",
    "",
    f"- Validated RGB pairs: {EXPECTED_TOTAL}",
    f"- Development pairs: {EXPECTED_DEVELOPMENT}",
    f"- Final test pairs: {EXPECTED_TEST}",
    f"- Model: {MODEL_WEIGHTS}",
    f"- Epochs: {EPOCHS}",
    f"- Image size: {IMAGE_SIZE}",
    f"- Batch size: {BATCH_SIZE}",
    "",
    "The split is grouped by collection context. The two final test contexts were kept untouched until final evaluation.",
]

if not cv_results_df.empty:
    summary_lines += [
        "",
        "## Grouped CV mean",
        f"- Precision: {cv_results_df['precision'].mean():.4f}",
        f"- Recall: {cv_results_df['recall'].mean():.4f}",
        f"- mAP@50: {cv_results_df['mAP50'].mean():.4f}",
        f"- mAP@50:95: {cv_results_df['mAP50_95'].mean():.4f}",
    ]

if final_metrics is not None:
    summary_lines += [
        "",
        "## Final test",
        f"- Precision: {float(final_metrics.box.mp):.4f}",
        f"- Recall: {float(final_metrics.box.mr):.4f}",
        f"- mAP@50: {float(final_metrics.box.map50):.4f}",
        f"- mAP@50:95: {float(final_metrics.box.map):.4f}",
    ]

(RESULTS_ROOT / "e0_rgb_summary.md").write_text(
    "\n".join(summary_lines) + "\n",
    encoding="utf-8",
)

print((RESULTS_ROOT / "e0_rgb_summary.md").read_text(encoding="utf-8"))


In [ ]:
# Final integrity checks.

assert len(pairs) == EXPECTED_TOTAL
assert int((pairs["partition"] == "development").sum()) == EXPECTED_DEVELOPMENT
assert int((pairs["partition"] == "test").sum()) == EXPECTED_TEST
assert set(pairs.loc[pairs["partition"] == "test", "context"]) == TEST_CONTEXTS
assert set(pairs.loc[pairs["partition"] == "development", "context"]) == DEVELOPMENT_CONTEXTS

for fold_id, spec in folds.items():
    train_contexts = set(spec["train"])
    val_context = {spec["validation"]}
    assert train_contexts.isdisjoint(val_context)

print("E0 RGB BASELINE PIPELINE: READY / COMPLETE")
print("Validated pairs:", len(pairs))
print("Development pairs:", EXPECTED_DEVELOPMENT)
print("Final test pairs:", EXPECTED_TEST)
print("Outputs:", RESULTS_ROOT)
